# AURORA-VISION — Edge Deployment with ONNX + TensorRT

This notebook demonstrates the **export and optimization** pipeline for deploying
AURORA-VISION models on edge devices:

1. Export CLIP ViT-L/14 to ONNX (opset 17)
2. Export Swin Transformer to ONNX
3. INT8 quantization via TensorRT optimizer
4. Latency profiling on CPU and (optionally) GPU
5. Accuracy validation — PyTorch vs. ONNX diff < 1e-4

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv("../.env")

import torch
import numpy as np
import matplotlib.pyplot as plt

from utils.logger import get_logger
from utils.seed import set_seed

logger = get_logger("notebook.deployment")
set_seed(42)

EXPORT_DIR = "/tmp/aurora_onnx"
os.makedirs(EXPORT_DIR, exist_ok=True)
print(f"Export directory: {EXPORT_DIR}")

## 1. Export CLIP ViT-L/14 → ONNX

In [ ]:
from visual.clip_encoder import CLIPFrameEncoder
from deployment.onnx_exporter import ONNXExporter

clip_encoder = CLIPFrameEncoder()
exporter = ONNXExporter(output_dir=EXPORT_DIR)

# Export CLIP visual encoder
clip_onnx_path = exporter.export_clip(
    model=clip_encoder.model,
    save_path=os.path.join(EXPORT_DIR, "clip_vit_l14.onnx"),
)
print(f"CLIP ONNX saved: {clip_onnx_path}")
print(f"File size: {os.path.getsize(clip_onnx_path) / 1e6:.1f} MB")

## 2. Export Swin Transformer → ONNX

In [ ]:
from visual.swin_encoder import SwinEncoder

swin_encoder = SwinEncoder()
swin_onnx_path = exporter.export_swin(
    model=swin_encoder.model,
    save_path=os.path.join(EXPORT_DIR, "swin_transformer.onnx"),
)
print(f"Swin ONNX saved: {swin_onnx_path}")
print(f"File size: {os.path.getsize(swin_onnx_path) / 1e6:.1f} MB")

## 3. ONNX Validation — PyTorch vs ONNX Runtime

In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(clip_onnx_path, providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name

dummy_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

# ONNX output
ort_out = sess.run(None, {input_name: dummy_input})[0]

# PyTorch output
with torch.no_grad():
    pt_out = clip_encoder.model.visual(torch.tensor(dummy_input)).cpu().numpy()

max_diff = np.abs(pt_out - ort_out).max()
print(f"Max absolute difference PyTorch vs ONNX: {max_diff:.2e}")
assert max_diff < 1e-4, f"Validation failed: max_diff={max_diff}"
print("✓ ONNX validation passed (max diff < 1e-4)")

## 4. Latency Profiling

In [ ]:
from deployment.latency_profiler import LatencyProfiler

profiler = LatencyProfiler()

# Profile CLIP on CPU
clip_profile = profiler.profile_model(
    model=clip_encoder.model.visual,
    input_shape=(1, 3, 224, 224),
    n_runs=50,
    device="cpu",
)
print("CLIP ViT-L/14 Latency (CPU):")
print(f"  Mean : {clip_profile['mean_ms']:.2f} ms")
print(f"  P50  : {clip_profile['p50_ms']:.2f} ms")
print(f"  P95  : {clip_profile['p95_ms']:.2f} ms")
print(f"  P99  : {clip_profile['p99_ms']:.2f} ms")

## 5. Latency Comparison Chart

In [ ]:
import plotly.graph_objects as go

models   = ["CLIP ViT-L/14", "Swin-T", "Whisper-large-v3", "TrOCR-large"]
mean_ms  = [
    clip_profile.get("mean_ms", 0),
    50,   # Example Swin-T latency
    1200, # Example Whisper latency
    80,   # Example TrOCR latency
]

fig = go.Figure(go.Bar(
    x=models,
    y=mean_ms,
    marker_color=["#00bcd4", "#4caf50", "#ff9800", "#9c27b0"],
    text=[f"{v:.0f} ms" for v in mean_ms],
    textposition="outside",
))
fig.update_layout(
    title="AURORA-VISION Model Inference Latency (CPU)",
    xaxis_title="Model",
    yaxis_title="Mean Latency (ms)",
    paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117",
    font=dict(color="white"),
)
fig.show()

## 6. TensorRT INT8 Optimization (requires NVIDIA GPU + TensorRT)

In [ ]:
if torch.cuda.is_available():
    from deployment.tensorrt_optimizer import TensorRTOptimizer

    optimizer = TensorRTOptimizer()
    trt_engine_path = optimizer.optimize(
        onnx_path=clip_onnx_path,
        precision="int8",
        output_path=os.path.join(EXPORT_DIR, "clip_int8.trt"),
    )
    print(f"TensorRT INT8 engine saved: {trt_engine_path}")
    print(f"Engine size: {os.path.getsize(trt_engine_path) / 1e6:.1f} MB")
else:
    print("No CUDA GPU detected — TensorRT optimization requires an NVIDIA GPU.")
    print("Run on a GPU instance (e.g., Colab T4 / A100) to enable TensorRT INT8.")